# Phase 3: YOLO Fine-Tuning

This notebook handles the fine-tuning of the YOLO model on our custom dataset using the RTX 3050 GPU.

In [1]:
# 1. Environment & GPU Verification
from ultralytics import YOLO
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is not available. Training will fall back to CPU and be very slow!")

PyTorch version: 2.7.1+cu118
CUDA available: True
Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU


### Model Initialization & Training

We will load a pretrained YOLOv8s (Small) or YOLO11s model and fine-tune it.
- **`device=0`**: explicitly forces the usage of your RTX 3050 GPU.
- **`batch=16`**: optimal for a 4GB/8GB VRAM GPU. Adjust to `8` if you get an Out of Memory (OOM) error.
- **`epochs=50`**: a good starting point for a custom dataset. YOLO implements Early Stopping by default if no improvement is seen.

In [3]:
# 2. Initialize and Train the Model
# If you prefer YOLO11s, just change 'yolov8s.pt' to 'yolo11s.pt'
model = YOLO('yolo26s.pt') 

# Start training! This will download the base weights automatically.
results = model.train(
    data='dataset_custom.yaml', # Points to our dataset configuration
    epochs=40,                  # Number of training epochs
    imgsz=640,                  # Default YOLO image size
    batch=16,                   # Batch size (reduce to 8 if doing Out of Memory on RTX 3050)
    device=0,                   # Uses the first CUDA GPU (RTX 3050)
    name='radar_yolo_model',    # Custom folder name where weights will be saved
    workers=4                   # Number of CPU workers for dataloading
)

Ultralytics 8.4.48  Python-3.11.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_custom.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=radar_yolo_model, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

### Validation

After training completes, the best weights will be saved automatically in the `runs/detect/radar_yolo_model/weights/best.pt` directory.
We will validate it against our validation split (`Dataset_Split/val`).

In [4]:
# 3. Model Validation
metrics = model.val() # Evaluates on the validation set specified in dataset_custom.yaml

print(f"\nmAP50-95: {metrics.box.map}") # Mean Average Precision over IoU thresholds 0.5 to 0.95
print(f"mAP50: {metrics.box.map50}")   # Mean Average Precision at IoU 0.5

Ultralytics 8.4.48  Python-3.11.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO26s summary (fused): 122 layers, 9,466,341 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 817.6209.1 MB/s, size: 58.1 KB)
val: Scanning D:\coding\Radar_Detection\Dataset_Split\val\1625227980720_jpg.rf.cache... 144 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 144/144  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 3.3it/s 2.7s0.3ss
                   all        144        457      0.887      0.714      0.754      0.665
                   car        137        201      0.952      0.731       0.75      0.727
             car_plate        130        131      0.977      0.986      0.989      0.897
                driver         53        125      0.732      0.424      0.523      0.371
Speed: 2.8ms preprocess, 12.4ms inference, 0.0ms loss, 0.1ms postprocess per image
Resu